In [1]:
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np

PROJECT_DIR = Path(r"G:\SEVA_Ai\SEVA_Ai")

GOLDEN_PATH = (
    PROJECT_DIR
    / "notebooks/golden_dataset/discovery/seva_discovery_golden_v1.jsonl"
)

DISCOVERY_DB_PATH = Path(r"G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\vectorstores\discovery")

MODEL_PATH = Path(
    r"G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2"
)

required_files = [
    "config.json",
    "model.safetensors",
    "modules.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.txt",
    "1_Pooling/config.json"
]

missing_files = [
    file for file in required_files
    if not (MODEL_PATH / file).exists()
]

if not missing_files:
    print("Model files verified successfully!")
    print("Model location:", MODEL_PATH)
else:
    print("Missing files:", missing_files)

Model files verified successfully!
Model location: G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2


In [2]:
import os
import time

os.environ["HF_HUB_OFFLINE"] = "1"

from langchain_huggingface import HuggingFaceEmbeddings

start_time = time.perf_counter()

embedding_model = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

load_time = time.perf_counter() - start_time

test_embedding = embedding_model.embed_query(
    "What government schemes are available for farmers?"
)

print(f"Model loading time: {load_time:.2f} seconds")
print("Embedding dimensions:", len(test_embedding))
print("Model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loading time: 254.25 seconds
Embedding dimensions: 384
Model loaded successfully!


In [3]:
from collections import defaultdict
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

VECTORSTORE_DIR = Path(r"G:\SEVA_Ai\SEVA_Ai\output\SEVA_CHILDREN\vectorstores")
golden_df = pd.read_json(GOLDEN_PATH, lines=True)


discovery_db = Chroma(
    collection_name="seva_discovery_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "discovery")
)

detail_db = Chroma(
    collection_name="seva_detail_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "detail")
)

discovery_count = discovery_db._collection.count()
detail_count = detail_db._collection.count()

print("Discovery collection:", discovery_count)
print("Detail collection:", detail_count)
print("Total indexed chunks:", discovery_count + detail_count)

Discovery collection: 23438
Detail collection: 67945
Total indexed chunks: 91383


## Dense Retrieval — Full Golden Dataset Tracing

Execute Dense retrieval across all 605 golden dataset queries. Generate one Phoenix span per query and preserve the existing retrieval results.

In [4]:
import time
import json
import pandas as pd

from phoenix.otel import register
from opentelemetry import trace
from opentelemetry.trace import Status, StatusCode

# Phoenix connection
tracer_provider = register(
    project_name="SEVA_AI",
    endpoint="http://localhost:6006/v1/traces",
    protocol="http/protobuf",
    batch=True
)

tracer = tracer_provider.get_tracer("seva_ai.retrieval")

print("Phoenix connected:", type(trace.get_tracer_provider()).__name__)

OpenTelemetry Tracing Details
|  Phoenix Project: SEVA_AI
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Phoenix connected: TracerProvider


In [7]:
RETRIEVAL_K = 30

def retrieve_schemes(query: str, k: int = RETRIEVAL_K):

    with tracer.start_as_current_span("dense_retrieval") as span:
        start = time.perf_counter()

        try:
            span.set_attribute("retrieval.group", "Dense_Retrieval")
            span.set_attribute("retrieval.method", "dense")
            span.set_attribute("retrieval.query", query)
            span.set_attribute("retrieval.top_k", k)

            results = discovery_db.similarity_search_with_score(
                query=query,
                k=k
            )

            ranked_schemes = []
            seen_scheme_ids = set()

            for chunk_rank, (doc, score) in enumerate(results, start=1):
                scheme_id = doc.metadata.get("scheme_id")

                if scheme_id and scheme_id not in seen_scheme_ids:
                    seen_scheme_ids.add(scheme_id)

                    ranked_schemes.append({
                        "scheme_id": scheme_id,
                        "rank": len(ranked_schemes) + 1,
                        "chunk_rank": chunk_rank,
                        "distance": float(score),
                        "section": doc.metadata.get("child_type")
                    })

            latency = time.perf_counter() - start

            span.set_attribute("retrieval.chunk_count", len(results))
            span.set_attribute("retrieval.unique_scheme_count", len(ranked_schemes))
            span.set_attribute("retrieval.latency_seconds", latency)
            span.set_attribute(
                "retrieval.top_scheme_ids",
                json.dumps([s["scheme_id"] for s in ranked_schemes[:10]])
            )

            span.set_status(Status(StatusCode.OK))

            return {
                "query": query,
                "retrieved_chunks": len(results),
                "retrieved_schemes": ranked_schemes,
                "latency": latency
            }

        except Exception as exc:
            span.record_exception(exc)
            span.set_status(Status(StatusCode.ERROR, str(exc)))
            raise

In [8]:
dense_results = []

for i, (_, row) in enumerate(golden_df.iterrows(), start=1):

    result = retrieve_schemes(row["question"])

    dense_results.append({
        "test_id": row["test_id"],
        "question": row["question"],
        "relevant_scheme_ids": row["relevant_scheme_ids"],
        "ranked_scheme_ids": [
            s["scheme_id"] for s in result["retrieved_schemes"]
        ],
        "retrieved_schemes": result["retrieved_schemes"],
        "retrieved_chunks": result["retrieved_chunks"],
        "latency": result["latency"]
    })

    if i % 50 == 0:
        tracer_provider.force_flush()
        print(f"Processed {i}/{len(golden_df)}")

tracer_provider.force_flush()

Processed 50/605
Processed 100/605
Processed 150/605
Processed 200/605
Processed 250/605
Processed 300/605
Processed 350/605
Processed 400/605
Processed 450/605
Processed 500/605
Processed 550/605
Processed 600/605


True

## BM25 Retrieval — Phoenix Instrumentation

Execute BM25 retrieval for all golden dataset queries. Deduplicate retrieved chunks into ranked schemes and capture query-level latency, retrieved chunks, unique schemes, and errors using Phoenix custom spans.

In [10]:
import re
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

# Fetch all discovery documents from Chroma
discovery_data = discovery_db.get(
    include=["documents", "metadatas"]
)

# Convert Chroma records into LangChain documents
discovery_documents = [
    Document(
        page_content=content,
        metadata=metadata or {}
    )
    for content, metadata in zip(
        discovery_data["documents"],
        discovery_data["metadatas"]
    )
    if content and metadata and metadata.get("scheme_id")
]

# BM25 tokenizer
def bm25_tokenize(text):
    return re.findall(r"\w+", text.lower())

# Build BM25 index
bm25_retriever = BM25Retriever.from_documents(
    discovery_documents,
    k=30,
    preprocess_func=bm25_tokenize
)

print("Chroma documents:", len(discovery_data["documents"]))
print("BM25 indexed documents:", len(discovery_documents))
print("Unique schemes:", len({
    doc.metadata["scheme_id"]
    for doc in discovery_documents
}))
print("BM25 retrieval depth:", bm25_retriever.k)

C:\Users\devendra\AppData\Local\Temp\ipykernel_14440\2570330136.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Chroma documents: 23438
BM25 indexed documents: 23438
Unique schemes: 4428
BM25 retrieval depth: 30


In [11]:
import time
import json
import pandas as pd

from opentelemetry.trace import Status, StatusCode

BM25_K = 30

def retrieve_bm25(query: str, k: int = BM25_K):

    with tracer.start_as_current_span("bm25_retrieval") as span:

        start = time.perf_counter()

        try:
            span.set_attribute("retrieval.method", "bm25")
            span.set_attribute("retrieval.query", query)
            span.set_attribute("retrieval.top_k", k)

            # Execute BM25 retrieval
            results = bm25_retriever.invoke(query)

            # Deduplicate chunks into ranked schemes
            ranked_schemes = []
            seen_scheme_ids = set()

            for chunk_rank, doc in enumerate(results, start=1):

                scheme_id = doc.metadata.get("scheme_id")

                if scheme_id and scheme_id not in seen_scheme_ids:
                    seen_scheme_ids.add(scheme_id)

                    ranked_schemes.append({
                        "scheme_id": scheme_id,
                        "rank": len(ranked_schemes) + 1,
                        "chunk_rank": chunk_rank,
                        "section": doc.metadata.get("child_type")
                    })

            latency = time.perf_counter() - start

            # Phoenix span attributes
            span.set_attribute("retrieval.chunk_count", len(results))
            span.set_attribute(
                "retrieval.unique_scheme_count",
                len(ranked_schemes)
            )
            span.set_attribute("retrieval.latency_seconds", latency)
            span.set_attribute(
                "retrieval.top_scheme_ids",
                json.dumps([
                    item["scheme_id"]
                    for item in ranked_schemes[:10]
                ])
            )

            span.set_status(Status(StatusCode.OK))

            return {
                "retrieved_chunks": len(results),
                "ranked_scheme_ids": [
                    item["scheme_id"] for item in ranked_schemes
                ],
                "retrieved_schemes": ranked_schemes,
                "latency": latency
            }

        except Exception as exc:
            span.record_exception(exc)
            span.set_status(Status(StatusCode.ERROR, str(exc)))
            raise


# Execute BM25 for the golden dataset
bm25_results = []

for _, row in golden_df.iterrows():

    result = retrieve_bm25(row["question"], k=BM25_K)

    bm25_results.append({
        "test_id": row["test_id"],
        "question": row["question"],
        "relevant_scheme_ids": row["relevant_scheme_ids"],
        "ranked_scheme_ids": result["ranked_scheme_ids"],
        "retrieved_schemes": result["retrieved_schemes"],
        "retrieved_chunks": result["retrieved_chunks"],
        "latency": result["latency"]
    })

bm25_df = pd.DataFrame(bm25_results)

print("BM25 retrieval completed.")
print("Total queries:", len(bm25_df))
print(
    "Average latency:",
    round(bm25_df["latency"].mean(), 4),
    "seconds"
)
print(
    "Average unique schemes:",
    round(bm25_df["ranked_scheme_ids"].apply(len).mean(), 2)
)

display(bm25_df.head(3))

BM25 retrieval completed.
Total queries: 605
Average latency: 0.6706 seconds
Average unique schemes: 24.16


,test_id,question,relevant_scheme_ids,ranked_scheme_ids,retrieved_schemes,retrieved_chunks,latency
0,DISC-0001,I need a loan to purchase a light motor vehicl...,[3he_F5gB5uF80_MGg5Hz],"[3he_F5gB5uF80_MGg5Hz, GMZscpoB0AGc0F0iyj3X, h...","[{'scheme_id': '3he_F5gB5uF80_MGg5Hz', 'rank':...",30,1.614269
1,DISC-0002,Looking for financing to buy a small van for g...,[3he_F5gB5uF80_MGg5Hz],"[B8a5F5gB0AGc0F0iwzWy, Qsa7F5gB0AGc0F0iVTVe, o...","[{'scheme_id': 'B8a5F5gB0AGc0F0iwzWy', 'rank':...",30,0.733457
2,DISC-0003,Seeking funds to acquire an auto‑rickshaw to s...,[3he_F5gB5uF80_MGg5Hz],"[caJdDZwBXu-iUKXq_Ubo, GBhS0ZwB5uF80_MGu2Cf, r...","[{'scheme_id': 'caJdDZwBXu-iUKXq_Ubo', 'rank':...",30,0.745969


In [12]:
test_query = golden_df.iloc[0]["question"]

bm25_test_result = retrieve_bm25(test_query)

print("Chunks:", bm25_test_result["retrieved_chunks"])
print("Unique schemes:", len(bm25_test_result["retrieved_schemes"]))
print("Latency:", round(bm25_test_result["latency"], 4), "seconds")

tracer_provider.force_flush()

Chunks: 30
Unique schemes: 22
Latency: 0.8114 seconds


True

## BM25 Full-Dataset Tracing

Run BM25 retrieval for all 605 golden dataset questions and export the results for later analysis.

In [13]:
bm25_results = []

for i, row in enumerate(golden_df.itertuples(index=False), start=1):
    result = retrieve_bm25(row.question)

    bm25_results.append({
        "test_id": row.test_id,
        "question": row.question,
        "relevant_scheme_ids": row.relevant_scheme_ids,
        "ranked_scheme_ids": [
            item["scheme_id"]
            for item in result["retrieved_schemes"]
        ],
        "retrieved_schemes": result["retrieved_schemes"],
        "retrieved_chunks": result["retrieved_chunks"],
        "latency": result["latency"]
    })

    if i % 50 == 0:
        tracer_provider.force_flush()
        print(f"Processed {i}/{len(golden_df)}")

tracer_provider.force_flush()

bm25_df = pd.DataFrame(bm25_results)

bm25_df.to_json(
    RESULTS_DIR / "bm25_phoenix_results.jsonl",
    orient="records",
    lines=True
)

print("Completed:", len(bm25_df))
print("Average latency:", round(bm25_df["latency"].mean(), 4), "seconds")
print("Saved:", RESULTS_DIR / "bm25_phoenix_results.jsonl")

Processed 50/605
Processed 100/605
Processed 150/605
Processed 200/605
Processed 250/605
Processed 300/605
Processed 350/605
Processed 400/605
Processed 450/605
Processed 500/605
Processed 550/605
Processed 600/605


NameError: name 'RESULTS_DIR' is not defined